# 第3章　债券定价原理

[![在 Colab 打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/ch03_bond_pricing.ipynb) [![在 Binder 打开](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/ch03_bond_pricing.ipynb)

复现例3.1（溢价/平价/折价）、例3.2（零息债）、价格—收益率曲线、拉回面值，以及 QuantLib 对拍与国债/国开债案例。


In [ ]:
# 自举单元：在 Colab/Binder 上自动安装本书复用包 fi；本地运行时自动跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/albertandking/fixed-income.git', '/content/fi-book'], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/fi-book'], check=False)
    else:
        print('提示：请在仓库根目录执行 `uv sync` 后再运行本 notebook。')


In [ ]:
import numpy as np
from fi.cashflow import make_cashflows
from fi.pricing import price_bond
from fi import plotting
plotting.use_chinese_style()


## 例3.1　溢价 / 平价 / 折价（3 年期、年付息、票息 3%）


In [ ]:
cfs, ts = make_cashflows(coupon_rate=0.03, maturity=3, freq=1, face=100)
for y in (0.02, 0.03, 0.04):
    P = price_bond(cfs, ts, y, freq=1)
    state = '溢价' if P > 100 else ('平价' if abs(P-100) < 1e-9 else '折价')
    print(f'y={y:.0%}  P={P:9.4f}  {state}')


## 例3.2　零息债定价


In [ ]:
print('1Y 零息@2.5% :', round(price_bond([100], [1], 0.025, 1), 4))
print('3Y 零息@2.5% :', round(price_bond([100], [3], 0.025, 1), 4))


### 价格—收益率曲线（编程实验 6）

$P$ 是 $y$ 的减函数：利率涨、价格跌。


In [ ]:
ys = np.linspace(0.00, 0.08, 161)
ps = [price_bond(cfs, ts, yi, 1) for yi in ys]
fig, ax = plotting.new_axes()
ax.plot(ys * 100, ps)
ax.axhline(100, ls=':', color='gray'); ax.axvline(3, ls=':', color='gray')
ax.set_xlabel('到期收益率 y (%)'); ax.set_ylabel('价格')
ax.set_title('图3-1　价格—收益率关系（票息 3%，y=3% 时平价）')
fig.tight_layout()


### 拉回面值 pull-to-par（编程实验 7）

固定收益率，溢价债与折价债的价格随剩余期限缩短都收敛到面值 100。


In [ ]:
mats = np.arange(10, 0 - 1e-9, -1)            # 剩余期限 10 -> 0 年（整数步长）
def price_at(coupon, y, mat):
    if mat <= 0:
        return 100.0
    cf, t = make_cashflows(coupon, mat, freq=1, face=100)
    return price_bond(cf, t, y, 1)

prem = [price_at(0.04, 0.025, m) for m in mats]   # 票息4% > 收益率2.5% -> 溢价
disc = [price_at(0.015, 0.025, m) for m in mats]  # 票息1.5% < 收益率2.5% -> 折价

fig, ax = plotting.new_axes()
ax.plot(mats, prem, label='溢价债（票息4%, y=2.5%）')
ax.plot(mats, disc, label='折价债（票息1.5%, y=2.5%）')
ax.axhline(100, ls=':', color='gray')
ax.set_xlabel('剩余期限（年）'); ax.set_ylabel('价格')
ax.invert_xaxis()
ax.set_title('图3-2　拉回面值：到期临近，价格收敛到 100'); ax.legend()
fig.tight_layout()


## 3.7　QuantLib 对拍（编程实验 8）

把例3.1 的债在三种收益率下用 `fi.price_bond` 与 QuantLib `FixedRateBond` 定价，比较误差。


In [ ]:
import QuantLib as ql
today = ql.Date(15, 6, 2026)
ql.Settings.instance().evaluationDate = today
sched = ql.Schedule(today, today + ql.Period(3, ql.Years), ql.Period(ql.Annual),
                    ql.China(ql.China.IB), ql.Following, ql.Following,
                    ql.DateGeneration.Backward, False)
dc = ql.ActualActual(ql.ActualActual.ISDA)
bond = ql.FixedRateBond(0, 100.0, sched, [0.03], dc)

print(f"{'y':>5}{'fi.price_bond':>16}{'QuantLib':>14}{'误差':>12}")
for y in (0.02, 0.03, 0.04):
    p_fi = price_bond(cfs, ts, y, 1)
    p_ql = ql.BondFunctions.cleanPrice(bond, ql.InterestRate(y, dc, ql.Compounded, ql.Annual))
    print(f'{y:>5.0%}{p_fi:>16.4f}{p_ql:>14.4f}{abs(p_fi-p_ql):>12.2e}')


## 3.8　国债 vs 国开债定价


In [ ]:
cf10, t10 = make_cashflows(coupon_rate=0.0255, maturity=10, freq=1, face=100)
p_cgb = price_bond(cf10, t10, 0.0255, 1)   # 国债 @2.55%
p_cdb = price_bond(cf10, t10, 0.0275, 1)   # 国开债 同票息、@2.75%
print(f'10Y CGB @2.55% = {p_cgb:.4f}  （平价）')
print(f'10Y CDB @2.75% = {p_cdb:.4f}  （折价）')
print(f'20bp 利差对应价格差 = {p_cgb - p_cdb:.4f} 元/百元面值')


---

> 小结：`fi.pricing.price_bond` 与 QuantLib `FixedRateBond` 定价一致；价格随收益率反向变动、随到期临近拉回面值，利差直接体现在价格上。
